# Assignment — surrogate extrapolation & uncertainty

Builds on `notebooks/ddm_heat_surrogate.ipynb` and `ddm_uncertainty.ipynb`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..','..')))
import numpy as np, torch, matplotlib.pyplot as plt
from pyutils import pinn_core as pc
from pyutils.pinn_core import MLP
torch.set_default_dtype(torch.float64); torch.manual_seed(0); np.random.seed(0)
pc.set_plot_style()
T=lambda x,t,a: np.sin(np.pi*x)*np.exp(-a*t)
a_train=np.array([1.0,1.5,2.0,2.5,3.0]); xs=np.linspace(0,1,21); ts=np.linspace(0,1,21)
def make_data(a_list):
    X=[];Y=[]
    for a in a_list:
        for x in xs:
            for t in ts: X.append([x,t,a]); Y.append(T(x,t,a))
    return np.array(X), np.array(Y).reshape(-1,1)
X,Y=make_data(a_train); Xm,Xs=X.mean(0),X.std(0); Ym,Ys=Y.mean(),Y.std()
def train_surrogate(seed):
    torch.manual_seed(seed); net=MLP([3,64,64,64,1]); opt=torch.optim.Adam(net.parameters(),lr=2e-3)
    Xt=torch.tensor((X-Xm)/Xs); Yt=torch.tensor((Y-Ym)/Ys)
    for _ in range(3000):
        opt.zero_grad(); l=((net(Xt)-Yt)**2).mean(); l.backward(); opt.step()
    return net
def field_rmse(net,a):
    xg,tg=np.meshgrid(xs,ts); P=np.c_[xg.ravel(),tg.ravel(),np.full(xg.size,a)]
    with torch.no_grad(): pr=net(torch.tensor((P-Xm)/Xs)).numpy().ravel()*Ys+Ym
    return float(np.sqrt(np.mean((pr-T(xg.ravel(),tg.ravel(),a))**2)))

## Task 1 — quantify the extrapolation cliff

Train the surrogate and show the field RMSE at the extrapolation point $a=4$ is at least 5× the in-range RMSE at $a=2$.

In [2]:
ratio=None
### BEGIN SOLUTION
net=train_surrogate(0)
ratio=field_rmse(net,4.0)/field_rmse(net,2.0)
### END SOLUTION
print(f'RMSE(a=4)/RMSE(a=2) = {ratio:.1f}')
assert ratio is not None and ratio > 5

RMSE(a=4)/RMSE(a=2) = 11.2


## Task 2 — uncertainty flags the unreliable region

Train a 3-model ensemble and show the prediction spread (std, averaged over the field) at $a=4$ exceeds that at $a=2$ — i.e. uncertainty grows where the surrogate extrapolates.

In [3]:
std4=std2=None
### BEGIN SOLUTION
ens=[train_surrogate(s) for s in range(3)]
xg,tg=np.meshgrid(xs,ts)
def preds(a):
    P=np.c_[xg.ravel(),tg.ravel(),np.full(xg.size,a)]
    return np.stack([(m(torch.tensor((P-Xm)/Xs)).detach().numpy().ravel()*Ys+Ym) for m in ens])
std4=float(preds(4.0).std(0).mean()); std2=float(preds(2.0).std(0).mean())
### END SOLUTION
print(f'ensemble std  a=2:{std2:.3e}  a=4:{std4:.3e}')
assert std4 > std2

ensemble std  a=2:6.070e-04  a=4:7.032e-03
